In [ ]:
# Llama-3.2-1B is a gated model.
# In THIS Kaggle notebook, add/enable a Secret named HF_TOKEN before running.
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    raise RuntimeError(
        "HF_TOKEN is not available to this Kaggle notebook. "
        "Open Add-ons > Secrets, add/enable a secret named HF_TOKEN, "
        "then restart this run."
    ) from exc

if not HF_TOKEN or not HF_TOKEN.startswith("hf_"):
    raise RuntimeError("HF_TOKEN was found, but it does not look like a valid Hugging Face token.")

# Authenticate once for Hugging Face Hub calls.
login(token=HF_TOKEN, add_to_git_credential=False)
print("Hugging Face authentication: OK")


In [ ]:
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from transformers import AutoModelForCausalLM
import matplotlib.pyplot as plt

RESULT_ROOT = Path("/kaggle/working/Assignment1")
Q4_FIGURE_DIR = RESULT_ROOT / "q4_figures"
Q4_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "meta-llama/Llama-3.2-1B"

# Question 4 must analyze the ORIGINAL Llama-3.2-1B model only.
# Pass the token explicitly so gated-model access does not depend on cached login state.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
)
model.eval()

print("Loaded original model:", MODEL_ID)
print("Q4 figures will be saved to:", Q4_FIGURE_DIR)


In [ ]:
# Question 4
# Analyze q_proj weights of Layers 0, 7, and 15 in the ORIGINAL model.
# No fine-tuning or evaluation is performed here.

LAYERS = [0, 7, 15]
ENERGY_THRESHOLD = 50.0
PLOT_COUNT = 200

q4_results = []

for k in LAYERS:
    # Query-projection weight specified by the assignment.
    q_weight = model.model.layers[k].self_attn.q_proj.weight.detach()

    # Full SVD; no target rank is specified for Question 4.
    U, S, Vh = torch.linalg.svd(
        q_weight.float(),
        full_matrices=False,
    )

    # Cumulative retained energy (%), based on squared singular values.
    energy_ratios = (
        torch.cumsum(S ** 2, dim=0)
        / torch.sum(S ** 2)
        * 100
    )

    # Minimum rank whose cumulative energy reaches at least 50%.
    threshold_indices = torch.nonzero(
        energy_ratios >= ENERGY_THRESHOLD,
        as_tuple=False,
    )
    if len(threshold_indices) == 0:
        min_rank_50 = len(S)
    else:
        min_rank_50 = int(threshold_indices[0].item()) + 1

    q4_results.append({
        'Layer': k,
        'Minimum Target Rank for >=50% Energy': min_rank_50,
        'Cumulative Energy at Minimum Rank (%)': float(
            energy_ratios[min_rank_50 - 1].item()
        ),
    })

    # The assignment asks to visualize only the first 200 singular values.
    n_plot = min(PLOT_COUNT, len(S))
    x = np.arange(1, n_plot + 1)
    s_plot = S[:n_plot].detach().cpu().numpy()
    e_plot = energy_ratios[:n_plot].detach().cpu().numpy()

    # (i) Cumulative energy (%) line chart.
    plt.figure(figsize=(8, 5))
    plt.plot(x, e_plot)
    plt.axhline(50, linestyle='--', linewidth=1)
    if min_rank_50 <= n_plot:
        plt.axvline(min_rank_50, linestyle='--', linewidth=1)
    plt.xlabel('Target Rank')
    plt.ylabel('Cumulative Energy (%)')
    plt.title(f'Layer {k} q_proj: Cumulative Energy (First {n_plot} Singular Values)')
    plt.tight_layout()

    energy_path = Q4_FIGURE_DIR / f'layer_{k}_cumulative_energy.png'
    plt.savefig(energy_path, dpi=200, bbox_inches='tight')
    plt.show()

    # (ii) Singular-value magnitudes bar chart.
    plt.figure(figsize=(8, 5))
    plt.bar(x, s_plot)
    plt.xlabel('Singular Value Index')
    plt.ylabel('Singular Value Magnitude')
    plt.title(f'Layer {k} q_proj: Singular Values (First {n_plot})')
    plt.tight_layout()

    singular_path = Q4_FIGURE_DIR / f'layer_{k}_singular_values.png'
    plt.savefig(singular_path, dpi=200, bbox_inches='tight')
    plt.show()

    print(
        f'Layer {k}: minimum rank for >=50% cumulative energy = '
        f'{min_rank_50}'
    )

Q4_TABLE = pd.DataFrame(q4_results)
print('\nQuestion 4 - Minimum Target Rank for at least 50% Cumulative Energy')
display(Q4_TABLE.round(4))

q4_table_path = RESULT_ROOT / 'q4_minimum_rank_50pct.csv'
Q4_TABLE.to_csv(q4_table_path, index=False)

print('\nSaved Q4 table:', q4_table_path)
print('Saved 6 Q4 charts under:', Q4_FIGURE_DIR)


In [ ]:
# Verify the six required Question 4 figures were created.
expected_q4_figures = [
    Q4_FIGURE_DIR / f'layer_{k}_{kind}.png'
    for k in LAYERS
    for kind in ['cumulative_energy', 'singular_values']
]

print('Question 4 figure checklist:')
for path in expected_q4_figures:
    print(('OK   ' if path.exists() else 'MISS '), path)

assert len(expected_q4_figures) == 6
assert all(path.exists() for path in expected_q4_figures), (
    'One or more required Question 4 figures are missing.'
)
